# Start here — contributor guide

Onboarding for the team building Phases 3+. By the end you'll have the notebooks running against live data and know how to plug in your own agent.

## 1. One-time setup

```bash
# from the repo ROOT (where docker-compose.yml and .env live)
cp .env.example .env            # GROQ_API_KEY optional offline
docker compose up -d postgres   # start the database

# the Python project lives in backend/ — run uv from there
cd backend
uv sync --group notebooks       # installs deps incl. Jupyter
uv run jupyter lab              # launch JupyterLab
```

In JupyterLab, pick the **Python 3 (ipykernel)** kernel (the backend venv).

## 2. Reading order

1. **`00_orchestration`** — see the whole pipeline and the state handoff.
2. **`10`–`50`** — one notebook per agent; your dev surface.
3. **`60_ingestion`** — how signals are fetched, gated, and stored.
4. **this notebook** — setup + adding a new agent.

## Setup — Postgres + ingest signals

Start Postgres from the **repo root** (`docker compose up -d postgres`), then these cells
wait for it and load real signals so the rest of the notebook runs against live data.
**Prerequisite:** Docker Desktop running.

> Prefer no database? Skip this section — the agent notebooks run fully offline on
> synthetic sample state.

In [ ]:
# Postgres is managed by `docker compose` from the REPO ROOT (where `.env` lives).
# Start it there first — in a terminal at the repo root:
#     docker compose up -d postgres
# (Already ran `docker compose up`? It's running — just continue.)
# The next cell waits until the database is reachable.
print("Ensure Postgres is up: run `docker compose up -d postgres` from the repo root.")

In [ ]:
import time

from agentic_scd.config import get_settings
from agentic_scd.db import ping

# get_settings() is cached; clear it each poll so a freshly-started DB is picked up.
deadline = time.time() + 60
get_settings.cache_clear()
status = ping()
while not status and time.time() < deadline:
    time.sleep(2)
    get_settings.cache_clear()
    status = ping()

print(status.detail)
if not status:
    print(
        "\n[!] Postgres isn't reachable. Is Docker Desktop running, and did "
        "`docker compose up -d postgres` succeed? You can still run the agent "
        "notebooks offline on synthetic sample state."
    )

In [ ]:
# Seed historical baselines (Freightos snapshot + Kaggle SupplyChainNet) — one-shot.
!uv run agentic-scd-batch

In [ ]:
# Run every enabled connector once (RSS + Open-Meteo + synthetic) through the pipeline.
!uv run agentic-scd-collect

In [ ]:
import psycopg

from agentic_scd.db import DatabaseNotConfiguredError, connect

try:
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            "SELECT status, count(*) FROM signals GROUP BY status ORDER BY status"
        )
        rows = cur.fetchall()
    if rows:
        print("signals by status:")
        for status_value, n in rows:
            print(f"  {status_value:>12}: {n}")
    else:
        print("signals table is empty — re-run the collect/batch cells above.")
except (DatabaseNotConfiguredError, psycopg.OperationalError) as exc:
    print(f"No DB available ({exc}); skipping the row-count check.")

## 3. Committing notebooks — clear outputs first

We commit notebooks with **outputs cleared** (no `nbstripout` hook — it's a convention). Before `git add`:

- In JupyterLab: **Edit → Clear Outputs of All Cells**, then save, **or**
- From the shell:

```bash
uv run jupyter nbconvert --clear-output --inplace notebooks/*.ipynb
```

This keeps diffs small and reviewable.

## 4. Add your own agent to the graph

An agent is just a function `state -> {channel: value}` wired into the pipeline. Three steps:

**a. Declare its result on the state** (`graph/state.py`)
```python
class GraphState(TypedDict, total=False):
    ...
    my_channel: list[str]   # what your agent writes
```

**b. Write the node** (`agents/my_agent.py`)
```python
def my_agent_node(state):
    signals = state.get('new_signals', [])
    return {'my_channel': [s.title for s in signals]}
```

**c. Wire it into the pipeline** (`graph/builder.py`): add a node constant, register it in `_NODE_FNS`, and place it in the `PIPELINE` list where it should run. Edges are derived from `PIPELINE` order automatically.

### Try the pattern here (no graph mutation)

A node is a plain function — you can develop it in isolation exactly like the per-agent notebooks, before wiring it into `builder.PIPELINE`.

In [ ]:
from agentic_scd.devtools import sample_state


def my_agent_node(state: dict) -> dict:
    # toy example: tag each signal with its title length
    signals = state.get("new_signals", [])
    return {"title_lengths": [len(s.title) for s in signals]}


st = sample_state(count=2)
print(my_agent_node(st))

When your node is ready, follow steps (a)–(c) above to add it to the graph, then re-run `00_orchestration` to watch it in the stream. Keep the `state -> dict` signature stable so the rest of the chain is unaffected.